# Module 6 — Code Along: CSV → API, the report is the product (the bank-accounts story, Day 2)

In **Module 5** we built an `AccountClient`: it speaks HTTP through one `_request` funnel, raises `APIError` on non-2xx, and returns the **Pydantic models from Module 4** instead of raw dicts. We learned the real `requests` shape with **zero network** by injecting a `FakeSession` — the same seam Day 3 mocks against.

**Today we feed that client a CSV of accounts and report what happened.** An operator hands us a file; we validate each row, send the good ones, and emit a JSON report that keeps validation errors, API errors, and successes in *separate buckets*. We also cover the plumbing real bulk loaders need: secrets from the environment, pagination, and rate-limit backoff.

Same canonical account shape and the same owners (Ada, Lin, Sam) as before:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

**No live server.** Files are written to a `tempfile` temp dir and are idempotent — we never touch the course `data/` folders. Run top to bottom.

## Setup — Module-5 recap (paste, don't narrate)

Same kit as Module 5 — notebooks can't import each other, so we paste it: the Pydantic models, `APIError`, `AccountClient` (with typed CRUD), and the SAME single configurable `FakeResponse`/`FakeSession`. **Nothing new here** — don't narrate these lines, just paste-and-run.

(The only addition over Module 5 is `update_account` for the chaining demo, and the `tags` field-validator that splits `"primary|online"` into a list — both are concepts Module 4 already taught.)

In [ ]:
from urllib.parse import urlsplit
from pydantic import BaseModel, Field, field_validator


class FakeResponse:
    """Mimics requests.Response: .status_code, .ok, .json(), .raise_for_status(), .headers."""
    def __init__(self, status_code, json_body, headers=None):
        self.status_code = status_code
        self._json = json_body
        self.headers = headers or {}        # e.g. {"Retry-After": "0.01"} for the 429 demo

    @property
    def ok(self):
        return self.status_code < 400

    def json(self):
        return self._json

    def raise_for_status(self):
        if not self.ok:                     # real requests raises requests.HTTPError here
            raise RuntimeError(f"HTTP {self.status_code}")


class FakeSession:
    """ONE configurable stand-in for requests.Session — no subclasses.

    routes maps "METHOD /path" -> a QUEUE (list) of outcomes consumed in order.
    Each outcome is a FakeResponse to return, OR an Exception to raise.
    A queue with one item left repeats it, so steady-state calls keep working.
    """
    def __init__(self, routes):
        self.routes = {k: (v if isinstance(v, list) else [v]) for k, v in routes.items()}
        self.headers = {}
        self.sent = []

    def request(self, method, url, **kwargs):
        path = urlsplit(url).path
        self.sent.append((method, path, kwargs))
        queue = self.routes[f"{method} {path}"]
        item = queue.pop(0) if len(queue) > 1 else queue[0]
        if isinstance(item, Exception):
            raise item
        return item


class APIError(Exception):
    """A non-2xx response, surfaced as a catchable error carrying status + detail."""
    def __init__(self, status, detail):
        super().__init__(f"{status}: {detail}")
        self.status = status
        self.detail = detail


# The Module 4 shapes, kept minimal. The tags validator (declared ONCE) splits "a|b|c" CSV strings.
class AccountBase(BaseModel):
    owner: str = Field(min_length=1)                 # blank owner is a validation error
    account_type: str = "savings"
    balance: float = Field(default=0.0, ge=0)        # negative balance is a validation error
    is_active: bool = True
    tags: list[str] = Field(default_factory=list)

    @field_validator("tags", mode="before")
    @classmethod
    def _split_tags(cls, v):
        # CSV carries tags as one 'primary|online' string; split it into a real list.
        if isinstance(v, str):
            return [t for t in v.split("|") if t]
        return v

class AccountCreate(AccountBase):
    id: int = Field(ge=1)

class BankAccount(AccountBase):
    id: int


class AccountClient:
    """Same client as Module 5: one _request funnel, APIError on non-2xx, typed CRUD."""
    def __init__(self, base_url, *, timeout=5.0, session=None):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self._session = session            # always a FakeSession in this notebook

    def _request(self, method, path, **kwargs):
        kwargs.setdefault("timeout", self.timeout)        # never hang forever
        resp = self._session.request(method, f"{self.base_url}{path}", **kwargs)
        if not resp.ok:                                   # 4xx/5xx -> raise APIError
            detail = resp.json().get("detail", "") if hasattr(resp, "json") else ""
            raise APIError(resp.status_code, detail)
        return resp

    def list_accounts(self) -> list[BankAccount]:
        data = self._request("GET", "/accounts").json()
        return [BankAccount.model_validate(r) for r in data]

    def create_account(self, payload: AccountCreate) -> BankAccount:
        data = self._request("POST", "/accounts", json=payload.model_dump()).json()
        return BankAccount.model_validate(data)

    def update_account(self, account_id: int, **changes) -> BankAccount:
        data = self._request("PATCH", f"/accounts/{account_id}", json=changes).json()
        return BankAccount.model_validate(data)

## Chaining: output of A → input of B

Real automation isn't one call — it's small clients **composed into a workflow**, where the output of step A is the input of step B. Below we chain `create → update → list`: the `id` the create returns is what the update and list use. Day 3 will mock each step independently — chaining is only safe because the seams are clean.

In [ ]:
# Chain create -> update -> list against a FakeSession (output of A feeds B).
session = FakeSession({
    "POST /accounts":     FakeResponse(201, {"id": 1, "owner": "Ada", "balance": 1500.0}),
    "PATCH /accounts/1":  FakeResponse(200, {"id": 1, "owner": "Ada", "balance": 1750.0}),
    "GET /accounts":      FakeResponse(200, [{"id": 1, "owner": "Ada", "balance": 1750.0}]),
})
client = AccountClient("http://localhost:8000", session=session)

created = client.create_account(AccountCreate(id=1, owner="Ada", balance=1500.0))
updated = client.update_account(created.id, balance=1750.0)   # created.id is the input to step B
everyone = client.list_accounts()                             # ...and step B's effect shows up in C
print(created.balance, "->", updated.balance)                # 1500.0 -> 1750.0
print("now on server:", [(a.owner, a.balance) for a in everyone])   # now on server: [('Ada', 1750.0)]

## Data-driven: CSV → API

An operator hands us a CSV. We walk it with `csv.DictReader`, validate each row through `AccountCreate.model_validate(row)`, then POST the survivors. Two failure modes are **fundamentally different** and must stay in separate buckets: a row that fails validation **never reaches the API** (the data is wrong), versus a row the **server rejects** with 409/422/500 (the request was well-formed but unacceptable). A third bucket holds successes. Pydantic coercion does the heavy lifting — CSV gives us strings, the model gives us typed values — and the `tags` validator turns `"primary|online"` into a real list.

In [ ]:
import csv
import os
import tempfile
from pydantic import ValidationError


# Write a tiny CSV to a TEMP dir (never the course data/ folders). Idempotent: same file each run.
TMP = tempfile.mkdtemp(prefix="m6-")
csv_path = os.path.join(TMP, "accounts.csv")
with open(csv_path, "w", newline="") as fh:
    fh.write(
        "id,owner,account_type,balance,tags\n"
        "1,Ada,savings,1500.0,primary|online\n"   # good: note tags is ONE string here
        "2,Lin,checking,300.0,\n"                  # good: empty tags -> []
        "3,,savings,500.0,primary\n"               # BAD: blank owner -> validation error
        "4,Sam,savings,-50.0,\n"                   # BAD: negative balance -> validation error
        "5,Ada,savings,200.0,vip\n"                # good shape, but server will 409 (dup owner+type)
    )

# Three SEPARATE buckets. Collapsing them would hide whether to fix the data or the server.
validated, validation_errors = [], []
with open(csv_path, newline="") as fh:
    # start=2 because row 1 is the header; this row number is what the operator sees in a spreadsheet.
    for row_no, row in enumerate(csv.DictReader(fh), start=2):
        try:
            payload = AccountCreate.model_validate(row)   # strings -> typed; blank/negative rejected
            validated.append((row_no, payload))
        except ValidationError as exc:
            validation_errors.append({"row": row_no, "input": row, "errors": exc.errors()})

print("validated rows:", [(n, p.owner, p.balance, p.tags) for n, p in validated])
# validated rows: [(2, 'Ada', 1500.0, ['primary', 'online']), (3, 'Lin', 300.0, []), (6, 'Ada', 200.0, ['vip'])]
print("validation errors at rows:", [e["row"] for e in validation_errors])   # validation errors at rows: [4, 5]
print("row 4 first error msg:", validation_errors[0]["errors"][0]["msg"])
# row 4 first error msg: String should have at least 1 character

## Environment & secrets

The base URL and token are **config, not code** — read them from the environment so the same script runs against staging or prod untouched. Use `os.environ["X"]` for **required** values (fail loud if missing) and `os.environ.get("X", default)` for optional ones. Workshops use a `.env` file + `python-dotenv`; production uses a secret manager that injects env vars. The one unbreakable rule: **never `print` the token** — it lands in CI logs forever. We set a *dummy* var in-notebook and only ever print a masked form.

In [ ]:
import os

# Required value: os.environ[...] raises if missing -> loud failure beats a silent wrong default.
# setdefault keeps this idempotent and avoids overwriting a real env var if one exists.
os.environ.setdefault("CATALOG_TOKEN", "s3cret-demo-token")
token = os.environ["CATALOG_TOKEN"]

# Optional value with a sensible default for local runs.
base_url = os.environ.get("CATALOG_BASE_URL", "http://localhost:8000")

# In a workshop you'd instead load these from a .env file:
#     from dotenv import load_dotenv; load_dotenv()
# In production a secret manager injects them; the code below is identical either way.

print("base_url:", base_url)                 # base_url: http://localhost:8000
print("token (masked):", token[:2] + "***")  # token (masked): s3***  <- NEVER print the raw token
print("have token:", bool(token))           # have token: True

## Pagination

A server won't return 10,000 accounts in one response — it pages them. Two shapes: **cursor** (the response carries a `next` link, e.g. `{"items": [...], "next": "/accounts?page=2"}`) and **offset** (the client sends `?offset=20&limit=10`). **Cursor is safer**: with offset, rows inserted or deleted mid-scan make you skip or double-read. The loop is the same idea — collect `items`, follow the pointer, stop when there's no `next`. Below, the same `FakeSession` serves two cursor pages.

In [ ]:
# A cursor-paginated server. Our FakeSession keys on the bare path (urlsplit drops the query),
# so we register one route per page path; each page's body carries the 'next' link to follow.
session = FakeSession({
    "GET /accounts": FakeResponse(200, {
        "items": [{"id": 1, "owner": "Ada"}, {"id": 2, "owner": "Lin"}],
        "next": "/accounts/page2",      # link to the next page (a path our FakeSession can route)
    }),
    "GET /accounts/page2": FakeResponse(200, {
        "items": [{"id": 3, "owner": "Sam"}],
        "next": None,                   # no next -> the loop stops here
    }),
})
client = AccountClient("http://localhost:8000", session=session)


def iter_all_accounts(client, start="/accounts"):
    """Follow the cursor: yield every item across pages until 'next' is None."""
    path = start
    while path:                                       # None ends the loop
        data = client._request("GET", path).json()    # transport faked; we just follow the link
        for item in data["items"]:
            yield item
        path = data.get("next")                       # advance the cursor to the next page


all_owners = [a["owner"] for a in iter_all_accounts(client)]
print("walked all pages ->", all_owners)   # walked all pages -> ['Ada', 'Lin', 'Sam']

## Rate limits & backoff

A `429 Too Many Requests` means *slow down* — and the server usually tells you how long via a `Retry-After` header. **Respect that header** instead of hammering; blind retries make the throttling worse. Wait the requested seconds, then retry the same request. In production you'd use **exponential backoff + jitter** so a whole fleet doesn't retry in lockstep. Below, the same `FakeSession` is queued to return `429` once (with a tiny `Retry-After` so the notebook stays fast), the client sleeps, then the retry succeeds.

In [ ]:
import time


def get_with_backoff(session, url, max_tries=3):
    """Honour Retry-After on 429 rather than blindly retrying."""
    for _ in range(max_tries):
        resp = session.request("GET", url)
        if resp.status_code == 429:
            delay = float(resp.headers.get("Retry-After", 1))   # read the server's instruction
            time.sleep(delay)                                   # tiny in the demo; real life = seconds
            continue                                            # then retry the SAME request
        return resp
    raise APIError(429, "still rate limited after retries")


# Queue the ONE FakeSession: 429 (with a near-zero Retry-After) then 200 — no subclass.
session = FakeSession({"GET /accounts": [
    FakeResponse(429, {"detail": "rate limited"}, headers={"Retry-After": "0.01"}),
    FakeResponse(200, [{"id": 1, "owner": "Ada"}]),
]})
resp = get_with_backoff(session, "http://localhost:8000/accounts")
print("succeeded after backoff ->", resp.json())   # succeeded after backoff -> [{'id': 1, 'owner': 'Ada'}]

## The report is the product

The CSV is input; the **report is the artifact** we hand back. We take the three buckets from the CSV pass — validation errors, then API outcomes — and assemble one JSON document: a `summary` of counts plus the detailed `validation_errors` and `api_errors` lists. That's what lets an operator fix *row 5* and re-run, and it's exactly what Day 3's tests will assert against. We send the validated rows through the real `AccountClient` (still against a `FakeSession`), catch `APIError` per row into its own bucket, then write `import_report.json` to the temp dir.

In [ ]:
import json


# Queue the ONE FakeSession with three scripted POST outcomes, consumed in order:
# rows 2 & 3 accepted (201), then the duplicate at row 6 (Ada/savings) rejected (409).
session = FakeSession({"POST /accounts": [
    FakeResponse(201, {"id": 1, "owner": "Ada", "balance": 1500.0, "tags": ["primary", "online"]}),
    FakeResponse(201, {"id": 2, "owner": "Lin", "balance": 300.0}),
    FakeResponse(409, {"detail": "owner Ada already has a savings account"}),
]})
client = AccountClient("http://localhost:8000", session=session)

# Send the rows that PASSED validation; sort API outcomes into created vs api_errors.
created, api_errors = [], []
for row_no, payload in validated:                       # 'validated' came from the CSV cell
    try:
        account = client.create_account(payload)
        created.append(account.id)
    except APIError as exc:                              # server rejected a well-formed request
        api_errors.append({"row": row_no, "status": exc.status, "detail": exc.detail})

# THE PRODUCT: one report, three buckets, ready to hand back (and for Day 3 to assert on).
report = {
    "summary": {
        "rows_read": len(validated) + len(validation_errors),
        "created": len(created),
        "validation_errors": len(validation_errors),
        "api_errors": len(api_errors),
    },
    "created": created,
    "validation_errors": validation_errors,
    "api_errors": api_errors,
}

report_path = os.path.join(TMP, "import_report.json")    # temp dir, not the course data/ folder
with open(report_path, "w") as fh:                       # overwrite each run -> idempotent
    json.dump(report, fh, indent=2)

print(json.dumps(report["summary"], indent=2))
# {
#   "rows_read": 5,
#   "created": 2,
#   "validation_errors": 2,
#   "api_errors": 1
# }
print("api_errors:", api_errors)         # api_errors: [{'row': 6, 'status': 409, 'detail': 'owner Ada already has a savings account'}]
print("wrote report ->", os.path.basename(report_path))   # wrote report -> import_report.json

## End of Day 2 — where this goes next

**Day 2, in one breath:** JSON + Pydantic models (M4) → an `AccountClient` that drives them over HTTP (M5) → a data-driven bulk loader that turns a CSV into a JSON **report**, keeping validation errors, API errors, and successes apart (M6). Along the way: secrets from the environment, cursor pagination, and `Retry-After` backoff. Every cell ran with **no live server** — the injected `FakeSession` is the whole reason that's possible.

**Tomorrow (Day 3) we TEST all of this.** That same injection seam becomes a `unittest.mock` / pytest fixture: we'll assert the client sends the right `json=`, that a 409 lands in `api_errors` and a blank owner lands in `validation_errors`, and that `import_report.json` has the exact shape above. The report we just built is the thing the tests pin down.

Next up: the **Lab 6** bulk-import workflow (for `Product`, against the real Day-1 FastAPI server), then Day 3's pytest suite, mocks, parametrize, and CI.